In [2]:
from datetime import datetime, timedelta
import requests
import pytz
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import duckdb
import time
import schedule

In [3]:
# Setup
analyzer = SentimentIntensityAnalyzer()
eastern = pytz.timezone('America/Toronto')

# Database connection
con = duckdb.connect('/Users/rimona/fintech-pulse/ingestion/fintech_pulse.duckdb')

# Companies and subreddits to track
companies = [
    "Wealthsimple", "Koho", "Questrade",
    "EQ Bank", "Neo Financial", "Mogo", 
    "Shakepay", "Float Financial", "Borrowell", "Nesto"
]

subreddits = [
    "PersonalFinanceCanada",
    "CanadaFinance",
    "investing_canada"
]

In [4]:
def fetch_by_field(company, subreddit, after_date, before_date, field, max_pages=3):
    
    url = "https://arctic-shift.photon-reddit.com/api/posts/search"
    headers = {"User-Agent": "fintech-pulse:v1.0 (by Rimona)"}
    
    all_posts = {}
    current_after = after_date
    page_count = 0
    
    while True:
        if page_count >= max_pages:
            break
        page_count += 1
        
        params = {
            "subreddit": subreddit,
            field: company,
            "after": current_after,
            "before": before_date,
            "limit": 100,
            "sort": "asc"
        }
        
        response = requests.get(url, params=params, headers=headers)
        time.sleep(1)
        data = response.json()
        
        if data.get('error'):
            print(f"Rate limited, skipping {company}...")
            break
        
        if not data.get('data'):
            break
        
        for item in data['data']:
            text = item.get('title', '') + " " + item.get('selftext', '')
            sentiment_score = analyzer.polarity_scores(text)['compound']
            created_utc = datetime.fromtimestamp(item['created_utc'], tz=eastern)
            
            post_id = item['id']
            all_posts[post_id] = {
                'post_id': post_id,
                'company': company,
                'subreddit': subreddit,
                'title': item.get('title', ''),
                'body': item.get('selftext', ''),
                'upvote_score': item.get('score', 0),
                'upvote_ratio': item.get('upvote_ratio', 0),
                'comment_count': item.get('num_comments', 0),
                'sentiment_score': sentiment_score,
                'created_utc': created_utc,
                'content_type': 'post'
            }
        
        if len(data['data']) < 100:
            break
        
        last_utc = data['data'][-1]['created_utc']
        current_after = str(last_utc + 1)
    
    return list(all_posts.values())


In [5]:
def fetch_posts(company, subreddit, after_date, before_date):
    by_title = fetch_by_field(company, subreddit, after_date, before_date, field="title")
    by_body = fetch_by_field(company, subreddit, after_date, before_date, field="selftext")
    
    combined = {}
    for post in by_title + by_body:
        combined[post['post_id']] = post
    
    return list(combined.values())


In [7]:
def fetch_comments(company, subreddit, after_date, before_date, max_pages=3):
    
    url = "https://arctic-shift.photon-reddit.com/api/comments/search"
    headers = {"User-Agent": "fintech-pulse:v1.0 (by Rimona)"}
    
    all_comments = {}
    current_after = after_date
    page_count = 0
    
    while True:
        if page_count >= max_pages:
            break
        page_count += 1
        
        params = {
            "subreddit": subreddit,
            "body": company,
            "after": current_after,
            "before": before_date,
            "limit": 100,
            "sort": "asc"
        }
        
        response = requests.get(url, params=params, headers=headers)
        time.sleep(1)
        data = response.json()
        
        if data.get('error'):
            print(f"Rate limited, skipping {company} comments...")
            break
        
        if not data.get('data'):
            break
        
        for item in data['data']:
            text = item.get('body', '')
            sentiment_score = analyzer.polarity_scores(text)['compound']
            created_utc = datetime.fromtimestamp(item['created_utc'], tz=eastern)
            
            comment_id = item['id']
            all_comments[comment_id] = {
                'post_id': comment_id,
                'company': company,
                'subreddit': subreddit,
                'title': '',
                'body': item.get('body', ''),
                'upvote_score': item.get('score', 0),
                'upvote_ratio': item.get('upvote_ratio', 0),
                'comment_count': 0,
                'sentiment_score': sentiment_score,
                'created_utc': created_utc,
                'content_type': 'comment'
            }
        
        if len(data['data']) < 100:
            break
        
        last_utc = data['data'][-1]['created_utc']
        current_after = str(last_utc + 1)
    
    return list(all_comments.values())

In [8]:
def insert_posts(results):
    if len(results) == 0:
        return
    
    now = datetime.now()
    
    for item in results:
        con.execute("""
            INSERT OR REPLACE INTO raw_posts (
                post_id,
                company,
                subreddit,
                title,
                body,
                upvote_score,
                upvote_ratio,
                comment_count,
                sentiment_score,
                created_utc,
                loaded_at,
                updated_at,
                content_type
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            item['post_id'],
            item['company'],
            item['subreddit'],
            item['title'],
            item['body'],
            item['upvote_score'],
            item['upvote_ratio'],
            item['comment_count'],
            item['sentiment_score'],
            item['created_utc'],
            now,
            now,
            item['content_type']
        ])
    
    print(f"Inserted/updated {len(results)} {results[0]['content_type']}s for {results[0]['company']}")

In [9]:
def run_daily():
    print(f"\nStarting daily load at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Only pull last 24 hours
    after_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
    before_date = datetime.now().strftime("%Y-%m-%d")
    
    for company in companies:
        for subreddit in subreddits:
            print(f"Fetching {company} from r/{subreddit}...")
            
            posts = fetch_posts(company, subreddit, after_date, before_date)
            insert_posts(posts)
            
            comments = fetch_comments(company, subreddit, after_date, before_date)
            insert_posts(comments)
    
    print(f"Daily load complete at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [10]:
# Schedule daily run at 9am
schedule.every().day.at("09:00").do(run_daily)

print("Daily load scheduler started. Waiting for 9am...")
print("Press Ctrl+C to stop.")

# Run once immediately on startup
run_daily()

# Then keep running on schedule
while True:
    schedule.run_pending()
    time.sleep(60)

Daily load scheduler started. Waiting for 9am...
Press Ctrl+C to stop.

Starting daily load at 2026-03-26 23:14:09
Fetching Wealthsimple from r/PersonalFinanceCanada...
Inserted/updated 5 posts for Wealthsimple
Inserted/updated 18 comments for Wealthsimple
Fetching Wealthsimple from r/CanadaFinance...
Fetching Wealthsimple from r/investing_canada...
Fetching Koho from r/PersonalFinanceCanada...
Fetching Koho from r/CanadaFinance...
Fetching Koho from r/investing_canada...
Fetching Questrade from r/PersonalFinanceCanada...
Inserted/updated 4 comments for Questrade
Fetching Questrade from r/CanadaFinance...
Fetching Questrade from r/investing_canada...
Fetching EQ Bank from r/PersonalFinanceCanada...
Inserted/updated 3 comments for EQ Bank
Fetching EQ Bank from r/CanadaFinance...
Rate limited, skipping EQ Bank...
Rate limited, skipping EQ Bank...
Rate limited, skipping EQ Bank comments...
Fetching EQ Bank from r/investing_canada...
Rate limited, skipping EQ Bank...
Fetching Neo Financial

KeyboardInterrupt: 